In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GPTQConfig

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
gptq_config = GPTQConfig(
    bits=4,
    dataset="wikitext2",
    tokenizer=tokenizer,
    group_size=128,
    damp_percent=0.01,
    desc_act=False,  # set to False can significantly speed up inference but the perplexity may slightly bad
    static_groups=False,
    sym=True,
    true_sequential=True,
    model_name_or_path=None,
    model_file_base_name="model",
)

# # Specify paths and hyperparameters for quantization
# quantize_config = BaseQuantizeConfig(
#     bits=4,
#     group_size=128,
#     damp_percent=0.01,
#     desc_act=False,  # set to False can significantly speed up inference but the perplexity may slightly bad
#     static_groups=False,
#     sym=True,
#     true_sequential=True,
#     model_name_or_path=None,
#     model_file_base_name="model",
# )

In [ ]:
# gptq_config = GPTQConfig(
#     bits=4,
#     dataset="c4",
#     tokenizer=tokenizer,
#     group_size=128,
#     damp_percent=0.01,
#     desc_act=False,  # set to False can significantly speed up inference but the perplexity may slightly bad
#     static_groups=False,
#     sym=True,
#     true_sequential=True,
#     model_name_or_path=None,
#     model_file_base_name="model",
# )
quantized_model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", quantization_config=gptq_config)

In [ ]:
save_dir = '/local_ssd2/nlyalyus/projects/nncf/examples/llm_compression/torch/qat_with_lora/out_Qwen_Qwen2_5-1_5B-Instruct_GPTQ_wiki'
quantized_model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

In [ ]:
import logging

import torch
from auto_gptq import BaseQuantizeConfig
from datasets import load_dataset
from torch import Tensor


def get_alpaca_with_chat(maxlen: int, tokenizer, device: torch.device) -> list[Tensor]:
    # dataset = load_dataset("databricks/databricks-dolly-15k", split="train", streaming=True)
    dataset = load_dataset("yahma/alpaca-cleaned", split="train", streaming=True)
    dataset = dataset.shuffle(seed=42, buffer_size=1000)  # Shuffle a buffer

    attempts = 0
    trainloader = []
    total_num_tokens = 0
    for example in dataset:
        attempts += 1
        if total_num_tokens >= 128 * 1024:  # NUM_SAMPLES:  # If streaming, limit checks
            print(
                f"Collected {len(trainloader)} samples by checking {attempts} samples, Avg length: {total_num_tokens // len(trainloader)}"
            )
            break

        instruction = example.get("instruction", "")
        context = example.get("input", "")
        response = example.get("output", "")
        if not instruction or not response:
            print("Skipping empty instruction and response")
            continue

        prompt = instruction
        if context:
            prompt += f"\n\n{context.strip()}"
        text = f"### Instruction: {prompt}\n### Response: {response}"
        # messages = [
        #     {"role": "system", "content": "You are a helpful assistant."},
        #     {"role": "user", "content": prompt},
        #     {"role": "assistant", "content": response},
        # ]
        # text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        trainloader.append(text)
        model_inputs = tokenizer([text])
        input_ids = torch.tensor(model_inputs.input_ids, dtype=torch.long).to(device)
        # if attempts == 1:
        #     print(tokenizer.batch_decode(input_ids)[0])
        total_num_tokens += input_ids.numel()
        # trainloader.append(dict(input_ids=input_ids, attention_mask=input_ids.ne(tokenizer.pad_token_id).long().to(device)))

    return trainloader


In [ ]:
# TODO: return just list of strings!!!
data = get_alpaca_with_chat(maxlen=8192, tokenizer=tokenizer, device="cpu")

In [ ]:
data[0]

In [ ]:
# data = []
# for item in dataset:
#     data.append({name: d.to('cpu') for name, d in item.items()})

In [ ]:
for name, t in data[0].items():
    print(t.device, t.dtype)
    # print(t.ne(tokenizer.pad_token_id).long())#to(dtype=torch.long))

In [ ]:
gptq_config = GPTQConfig(
    bits=4,
    dataset=data,
    tokenizer=tokenizer,
    group_size=128,
    damp_percent=0.01,
    desc_act=False,  # set to False can significantly speed up inference but the perplexity may slightly bad
    static_groups=False,
    sym=True,
    true_sequential=True,
    model_name_or_path=None,
    model_file_base_name="model",
)
quantized_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cuda:0",
    max_memory={0: "80GiB"},
    quantization_config=gptq_config
)

In [ ]:
quantized_model.config.quantization_config.dataset = None
save_dir = '/local_ssd2/nlyalyus/projects/nncf/examples/llm_compression/torch/qat_with_lora/out_Qwen_Qwen2_5-1_5B-Instruct_GPTQ_chat_alpaca_128x1024_2'
quantized_model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

In [ ]:
dataset_path = '/local_ssd2/nlyalyus/projects/nncf/examples/llm_compression/torch/qat_with_lora/calib.txt'
with open(dataset_path, 'r') as f:
    text = f.readlines()

In [ ]:
import re

def extract_user_blocks(text):
    # This regex finds all blocks starting with <|im_start|>user up to the next <|im_start|>user or end of string
    pattern = r'(<\|im_start\|>user.*?)(?=<\|im_start\|>user|$)'
    return re.findall(pattern, text, re.DOTALL)

# Example usage:
# text = ...  # your input text
user_blocks = extract_user_blocks(''.join(text))

In [ ]:
len(user_blocks)
sum(tokenizer(block, return_tensors='pt').input_ids.numel() for block in user_blocks) / 115

In [ ]:
gptq_config = GPTQConfig(
    bits=4,
    dataset=data,
    tokenizer=tokenizer,
    group_size=128,
    damp_percent=0.01,
    desc_act=False,  # set to False can significantly speed up inference but the perplexity may slightly bad
    static_groups=False,
    sym=True,
    true_sequential=True,
    model_name_or_path=None,
    model_file_base_name="model",
)
quantized_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cuda:0",
    max_memory={0: "80GiB"},
    quantization_config=gptq_config
)

In [ ]:
quantized_model.config.quantization_config.dataset = None
save_dir = '/local_ssd2/nlyalyus/projects/nncf/examples/llm_compression/torch/qat_with_lora/out_Qwen_Qwen2_5-1_5B-Instruct_GPTQ_chat_alpaca_128x1024_2'
quantized_model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

In [ ]:
import argparse
import json
from typing import Dict
import logging

import torch
import transformers
from transformers import AutoTokenizer
from transformers.trainer_pt_utils import LabelSmoother
from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig
IGNORE_TOKEN_ID = LabelSmoother.ignore_index

import logging

import torch
from auto_gptq import BaseQuantizeConfig
from datasets import load_dataset
from torch import Tensor


def get_alpaca_with_chat(tokenizer, device: torch.device) -> list[Tensor]:
    # dataset = load_dataset("databricks/databricks-dolly-15k", split="train", streaming=True)
    dataset = load_dataset("yahma/alpaca-cleaned", split="train", streaming=True)
    dataset = dataset.shuffle(seed=42, buffer_size=1000)  # Shuffle a buffer

    attempts = 0
    trainloader = []
    total_num_tokens = 0
    for example in dataset:
        attempts += 1
        if attempts ==755: # >= 128 * 1024:  # NUM_SAMPLES:  # If streaming, limit checks
            print(
                f"Collected {len(trainloader)} samples by checking {attempts} samples, Avg length: {total_num_tokens // len(trainloader)}"
            )
            break

        instruction = example.get("instruction", "")
        context = example.get("input", "")
        response = example.get("output", "")
        if not instruction or not response:
            print("Skipping empty instruction and response")
            continue

        prompt = instruction
        if context:
            prompt += f"\n{context.strip()}"
        # text = f"### Instruction: {prompt}\n### Response: {response}"
        text = {
            "conversations": [
                {
                    "from": "user",
                    "value": prompt
                },
                {
                    "from": "assistant",
                    "value": response
                }
            ]
        }
        trainloader.append(text)
        # model_inputs = tokenizer([text])
        # input_ids = torch.tensor(model_inputs.input_ids, dtype=torch.long).to(device)
        # if attempts == 1:
        #     print(tokenizer.batch_decode(input_ids)[0])
        # total_num_tokens += input_ids.numel()
        # trainloader.append(dict(input_ids=input_ids, attention_mask=input_ids.ne(tokenizer.pad_token_id).long().to(device)))

    return trainloader

def preprocess(
    sources,
    tokenizer: transformers.PreTrainedTokenizer,
    max_len: int,
    system_message: str = "You are a helpful assistant."
) -> Dict:
    roles = {"user": "<|im_start|>user", "assistant": "<|im_start|>assistant"}

    im_start = 151644 #tokenizer.im_start_id
    im_end = 151645 #tokenizer.im_end_id
    nl_tokens = tokenizer('\n').input_ids
    _system = tokenizer('system').input_ids + nl_tokens
    _user = tokenizer('user').input_ids + nl_tokens
    _assistant = tokenizer('assistant').input_ids + nl_tokens

    # Apply prompt templates
    data = []
    # input_ids, targets = [], []
    for i, source in enumerate(sources):
        source = source["conversations"]
        if roles[source[0]["from"]] != roles["user"]:
            source = source[1:]

        input_id, target = [], []
        system = [im_start] + _system + tokenizer(system_message).input_ids + [im_end] + nl_tokens
        input_id += system
        target += [im_start] + [IGNORE_TOKEN_ID] * (len(system)-3) + [im_end] + nl_tokens
        assert len(input_id) == len(target)
        for j, sentence in enumerate(source):
            role = roles[sentence["from"]]
            _input_id = tokenizer(role).input_ids + nl_tokens + \
                tokenizer(sentence["value"]).input_ids + [im_end] + nl_tokens
            input_id += _input_id
            if role == '<|im_start|>user':
                _target = [im_start] + [IGNORE_TOKEN_ID] * (len(_input_id)-3) + [im_end] + nl_tokens
            elif role == '<|im_start|>assistant':
                _target = [im_start] + [IGNORE_TOKEN_ID] * len(tokenizer(role).input_ids) + \
                    _input_id[len(tokenizer(role).input_ids)+1:-2] + [im_end] + nl_tokens
            else:
                raise NotImplementedError
            target += _target
        assert len(input_id) == len(target)
        input_id = torch.tensor(input_id[:max_len], dtype=torch.long).unsqueeze(dim=0).cpu()
        target = torch.tensor(target[:max_len], dtype=torch.long).cpu()
        data.append(dict(input_ids=input_id, attention_mask=input_id.ne(tokenizer.pad_token_id).long().cpu()))

    return data

In [ ]:
convs = get_alpaca_with_chat(tokenizer, 'cpu')
trainloader = preprocess(convs, tokenizer, max_len=8196)

In [ ]:
trainloader[0]['input_ids'].shape

In [ ]:
gptq_config = GPTQConfig(
    bits=4,
    dataset=trainloader,
    tokenizer=tokenizer,
    group_size=128,
    damp_percent=0.01,
    desc_act=False,  # set to False can significantly speed up inference but the perplexity may slightly bad
    static_groups=False,
    sym=False,
    true_sequential=True,
    model_name_or_path=None,
    model_file_base_name="model",
)
quantized_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cuda:0",
    max_memory={0: "80GiB"},
    quantization_config=gptq_config
)

In [ ]:
quantized_model.config.quantization_config.dataset = None
save_dir = '/local_ssd2/nlyalyus/projects/nncf/examples/llm_compression/torch/qat_with_lora/out_Qwen_Qwen2_5-1_5B-Instruct_GPTQ_chat_alpaca_755_manual_chat_asym'
quantized_model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

In [ ]:
ckpt_file = '/local_ssd2/nlyalyus/projects/nncf/examples/llm_compression/torch/qat_with_lora/out_Qwen_Qwen2_5-1_5B-Instruct_GPTQ_chat_alpaca_755_manual_chat'
model = AutoModelForCausalLM.from_pretrained(ckpt_file, device_map='cuda:0')

In [ ]:
for name, param in quantized_model.named_modules():
    # print(param.__class__.__name__, name)
    if param.__class__.__name__ == 'TritonV2QuantLinear':
        paramg_idx
        print(*dir(param), sep='\n')
        break
        # if param.zp.numel() == 0:
        #     print(param.zp.shape)


In [ ]:
print(param.scales.shape, param.g_idx.shape, param.qzeros.shape, param.qzero_format, param.qweight.shape)

In [ ]:
256 / 32

In [ ]:
ckpt_file = 'Qwen/Qwen2.5-1.5B-Instruct-GPTQ-Int4'
gptq = AutoModelForCausalLM.from_pretrained(ckpt_file, device_map='cuda:0')

In [ ]:
for name, param in gptq.named_modules():
    if param.__class__.__name__ == 'MarlinQuantLinear':
        if param.zp.numel() != 0:
            print(param.zp.shape)
        # print(*dir(param), sep="\n")
        # break

In [ ]:
save_dir = '/local_ssd2/nlyalyus/projects/nncf/examples/llm_compression/torch/qat_with_lora/gptq'
gptq.save_pretrained(save_dir)

In [ ]:
from lm_eval import simple_evaluate
from lm_eval.models.optimum_lm import HFLM
from pprint import pprint

task = "hellaswag"
lm_obj = HFLM(pretrained=gptq, batch_size=32)
results = simple_evaluate(lm_obj, tasks=[task], log_samples=False)
pprint(results["results"])

In [ ]:
from pathlib import Path
ckpt_file = '/local_ssd2/nlyalyus/projects/nncf/examples/llm_compression/torch/qat_with_lora/gptq' #/ 'last'# / "nncf_checkpoint.pth"
device = 'cuda'
gptq_disk = AutoModelForCausalLM.from_pretrained(ckpt_file, device_map=device)

task = "wikitext"
lm_obj = HFLM(pretrained=gptq_disk)
results = simple_evaluate(lm_obj, tasks=[task], log_samples=False)
pprint(results["results"])

In [256]:
def get_gsm8k(tokenizer) -> list[Tensor]:
    dataset = load_dataset("openai/gsm8k", 'main', split="train", streaming=True)
    dataset = dataset.shuffle(seed=42, buffer_size=1000)  # Shuffle a buffer

    attempts = 0
    trainloader = []
    total_num_tokens = 0
    for example in dataset:
        attempts += 1
        if total_num_tokens >= 128 * 1024:  # NUM_SAMPLES:  # If streaming, limit checks
            print(
                f"Collected {len(trainloader)} samples by checking {attempts} samples, Avg length: {total_num_tokens // len(trainloader)}"
            )
            break

        question = example.get("question", "")
        answer = example.get("answer", "")
        text = f"### Question: {question}\n### Answer: {answer}"
        trainloader.append(text)
        model_inputs = tokenizer([text])
        input_ids = torch.tensor(model_inputs.input_ids, dtype=torch.long)
        if attempts == 1:
            print(tokenizer.batch_decode(input_ids)[0])
        total_num_tokens += input_ids.numel()

    return trainloader


In [257]:
trainloader = get_gsm8k(tokenizer)

### Question: Krystian works in the library. He borrows an average of 40 books every day. Every Friday, his number of borrowed books is about 40% higher than the daily average. How many books does he borrow in a week if the library is open from Monday to Friday?
### Answer: The number of books borrowed on Friday is higher by 40 * 40/100 = <<40*40/100=16>>16 books.
There are 5 days from Monday to Friday inclusive, so Krystian borrows an average of 5 * 40 = <<5*40=200>>200 books during that time.
With Friday's increase in borrowings, during one week Krystian borrows 200 + 16 = <<200+16=216>>216 books.
#### 216
Collected 695 samples by checking 696 samples, Avg length: 188


In [258]:
gptq_config = GPTQConfig(
    bits=4,
    dataset=trainloader,
    tokenizer=tokenizer,
    group_size=128,
    damp_percent=0.01,
    desc_act=False,  # set to False can significantly speed up inference but the perplexity may slightly bad
    static_groups=False,
    sym=True,
    true_sequential=True,
    model_name_or_path=None,
    model_file_base_name="model",
)
quantized_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cuda:0",
    max_memory={0: "80GiB"},
    quantization_config=gptq_config
)

Detected gptqmodel and auto-gptq, will use gptqmodel


Quantizing model.layers blocks :   0%|          | 0/28 [00:00<?, ?it/s]

INFO:optimum.gptq.quantizer:Start quantizing block model.layers 1/28
INFO:optimum.gptq.quantizer:Module to quantize [['self_attn.q_proj'], ['self_attn.k_proj'], ['self_attn.v_proj'], ['self_attn.o_proj'], ['mlp.gate_proj'], ['mlp.up_proj'], ['mlp.down_proj']]


Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

INFO:optimum.gptq.quantizer:Quantizing self_attn.q_proj in block 1/28...
INFO:optimum.gptq.quantizer:Quantizing self_attn.k_proj in block 1/28...
INFO:optimum.gptq.quantizer:Quantizing self_attn.v_proj in block 1/28...
INFO:optimum.gptq.quantizer:Quantizing self_attn.o_proj in block 1/28...
INFO:optimum.gptq.quantizer:Quantizing mlp.gate_proj in block 1/28...
INFO:optimum.gptq.quantizer:Quantizing mlp.up_proj in block 1/28...
INFO:optimum.gptq.quantizer:Quantizing mlp.down_proj in block 1/28...
INFO:optimum.gptq.quantizer:Start quantizing block model.layers 2/28
INFO:optimum.gptq.quantizer:Module to quantize [['self_attn.q_proj'], ['self_attn.k_proj'], ['self_attn.v_proj'], ['self_attn.o_proj'], ['mlp.gate_proj'], ['mlp.up_proj'], ['mlp.down_proj']]


...

INFO:optimum.gptq.quantizer:Start quantizing block model.layers 28/28
INFO:optimum.gptq.quantizer:Module to quantize [['self_attn.q_proj'], ['self_attn.k_proj'], ['self_attn.v_proj'], ['self_attn.o_proj'], ['mlp.gate_proj'], ['mlp.up_proj'], ['mlp.down_proj']]


Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

INFO:optimum.gptq.quantizer:Quantizing self_attn.q_proj in block 28/28...
INFO:optimum.gptq.quantizer:Quantizing self_attn.k_proj in block 28/28...
INFO:optimum.gptq.quantizer:Quantizing self_attn.v_proj in block 28/28...
INFO:optimum.gptq.quantizer:Quantizing self_attn.o_proj in block 28/28...
INFO:optimum.gptq.quantizer:Quantizing mlp.gate_proj in block 28/28...
INFO:optimum.gptq.quantizer:Quantizing mlp.up_proj in block 28/28...
INFO:optimum.gptq.quantizer:Quantizing mlp.down_proj in block 28/28...
INFO:optimum.gptq.quantizer:Packing model...


INFO  Packing Kernel: Auto-selection: adding candidate `TritonV2QuantLinear`   


INFO:optimum.gptq.quantizer:model.layers.0.self_attn.k_proj
INFO:optimum.gptq.quantizer:model.layers.0.self_attn.o_proj
INFO:optimum.gptq.quantizer:model.layers.0.self_attn.q_proj
INFO:optimum.gptq.quantizer:model.layers.0.self_attn.v_proj
INFO:optimum.gptq.quantizer:model.layers.0.mlp.down_proj
INFO:optimum.gptq.quantizer:model.layers.0.mlp.gate_proj
INFO:optimum.gptq.quantizer:model.layers.0.mlp.up_proj
INFO:optimum.gptq.quantizer:model.layers.1.self_attn.k_proj
INFO:optimum.gptq.quantizer:model.layers.1.self_attn.o_proj
INFO:optimum.gptq.quantizer:model.layers.1.self_attn.q_proj
INFO:optimum.gptq.quantizer:model.layers.1.self_attn.v_proj
INFO:optimum.gptq.quantizer:model.layers.1.mlp.down_proj
INFO:optimum.gptq.quantizer:model.layers.1.mlp.gate_proj
INFO:optimum.gptq.quantizer:model.layers.1.mlp.up_proj
INFO:optimum.gptq.quantizer:model.layers.2.self_attn.k_proj
INFO:optimum.gptq.quantizer:model.layers.2.self_attn.o_proj
INFO:optimum.gptq.quantizer:model.layers.2.self_attn.q_proj
IN

In [259]:
task = "wikitext"
lm_obj = HFLM(pretrained=quantized_model)
results = simple_evaluate(lm_obj, tasks=[task], log_samples=False)
pprint(results["results"])

INFO:lm_eval.evaluator:Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
INFO:lm_eval.evaluator:Using pre-initialized model
INFO:lm_eval.api.task:Building contexts for wikitext on rank 0...
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 62/62 [00:00<00:00, 477.01it/s]
INFO:lm_eval.evaluator:Running loglikelihood_rolling requests
Running loglikelihood requests: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  9.31it/s]


{'wikitext': {'alias': 'wikitext',
              'bits_per_byte,none': 0.728815169608845,
              'bits_per_byte_stderr,none': 'N/A',
              'byte_perplexity,none': 1.6572774738270908,
              'byte_perplexity_stderr,none': 'N/A',
              'word_perplexity,none': 14.900647768936905,
              'word_perplexity_stderr,none': 'N/A'}}


In [278]:
task = "gsm8k"
lm_obj = HFLM(pretrained=quantized_model, batch_size=32)
results = simple_evaluate(lm_obj, tasks=[task], log_samples=False, apply_chat_template=True)
pprint(results["results"])

INFO:lm_eval.evaluator:Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
INFO:lm_eval.evaluator:Using pre-initialized model
INFO:lm_eval.api.task:Building contexts for gsm8k on rank 0...
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1319/1319 [00:06<00:00, 212.83it/s]
INFO:lm_eval.evaluator:Running generate_until requests
Running generate_until requests: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 1319/1319 [09:05<00:00,  2.42it/s]


{'gsm8k': {'alias': 'gsm8k',
           'exact_match,flexible-extract': 0.4533737680060652,
           'exact_match,strict-match': 0.10310841546626232,
           'exact_match_stderr,flexible-extract': 0.01371247104951545,
           'exact_match_stderr,strict-match': 0.008376436987507788}}


In [275]:
save_dir = '/local_ssd2/nlyalyus/projects/nncf/examples/llm_compression/torch/qat_with_lora/out_Qwen_Qwen2_5-1_5B-Instruct_GPTQ_gsm8k'
quantized_model.save_pretrained(save_dir)
# tokenizer.save_pretrained(save_dir)
# save_dir

In [265]:
results["config"]["model_dtype"] = str(results["config"]["model_dtype"])
with (Path(save_dir) / f'results_{task}.json').open('w') as f:
    json.dump(results, f, indent=4)

In [277]:
device = 'cuda'
# backend='triton'
gptq_disk = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct-GPTQ-Int4', device_map=device)#, quantization_config=GPTQConfig(bits=4, sym=True, group_size=128, backend=backend))

task = "wikitext"
lm_obj = HFLM(pretrained=gptq_disk)
results = simple_evaluate(lm_obj, tasks=[task], log_samples=False)
pprint(results["results"])

Detected gptqmodel and auto-gptq, will use gptqmodel


INFO   Kernel: Auto-selection: adding candidate `MarlinQuantLinear`            


INFO:lm_eval.evaluator:Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
INFO:lm_eval.evaluator:Using pre-initialized model
INFO:lm_eval.api.task:Building contexts for wikitext on rank 0...
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 62/62 [00:00<00:00, 493.01it/s]
INFO:lm_eval.evaluator:Running loglikelihood_rolling requests
Running loglikelihood requests: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  7.90it/s]


{'wikitext': {'alias': 'wikitext',
              'bits_per_byte,none': 0.7016623556721572,
              'bits_per_byte_stderr,none': 'N/A',
              'byte_perplexity,none': 1.6263777188096231,
              'byte_perplexity_stderr,none': 'N/A',
              'word_perplexity,none': 13.47398611206857,
              'word_perplexity_stderr,none': 'N/A'}}
